In [35]:
from torch import nn
import torch
import pandas as pd
from gensim.models import KeyedVectors
import gensim.downloader as api
from torch.nn.utils.rnn import pad_sequence
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

In [2]:
# !pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.7 MB/s eta 0:00:00


In [5]:
df = pd.read_csv('/content/sentimentdataset.csv')

In [6]:
df.head()

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19


In [7]:
df = df.set_index('Unnamed: 0')

In [8]:
X = df['Text']
Y= df['Sentiment']

In [9]:
glove = api.load("glove-wiki-gigaword-300")  # ~376MB

[==================================================] 100.0% 376.1/376.1MB downloaded


In [14]:
def tokenize(text):
    return text.lower().split()

In [15]:
def sentence_to_word_embeddings(sentence, model):
    tokens = tokenize(sentence)
    return [
        model[word]
        for word in tokens
        if word in model
    ]

In [17]:
X_embeddings = X.apply(lambda x: sentence_to_word_embeddings(x, glove))

In [20]:
# X_embeddings_tensors = torch.tensor( X_embeddings, dtype=torch.float32 )

X_tensors = [ torch.tensor(emb, dtype=torch.float32)   for emb in X_embeddings  ]

In [23]:
X_padded = pad_sequence( X_tensors, batch_first=True, padding_value=0.0 )

In [24]:
print(X_padded.shape)

torch.Size([732, 22, 300])


In [27]:
lengths = torch.tensor([t.shape[0] for t in X_tensors])

In [36]:

class SentimentBiLSTM(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=300,
            hidden_size=512,
            num_layers=4,
            batch_first=True,
            dropout=0.3,
            bidirectional=True,
            proj_size=128
        )

        # Because bidirectional + proj_size=128
        self.classifier = nn.Linear(2 * 128, num_classes)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, lengths):
        """
        x: (batch_size, max_seq_len, 300)
        lengths: (batch_size,)
        """

        # 1️⃣ Pack padded sequences
        packed_x = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # 2️⃣ LSTM forward
        packed_out, (h_n, c_n) = self.lstm(packed_x)

        # 3️⃣ Unpack
        out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )
        # out: (batch_size, max_seq_len, 256)

        # 4️⃣ Extract last valid timestep per sequence
        batch_size = out.size(0)
        last_outputs = out[
            torch.arange(batch_size),
            lengths - 1
        ]  # (batch_size, 256)

        # 5️⃣ Classification
        logits = self.classifier(self.dropout(last_outputs))

        return logits

In [30]:
help(nn.LSTM.__init__)

Help on function __init__ in module torch.nn.modules.rnn:

__init__(self, *args, **kwargs)
    Initialize internal Module state, shared by both nn.Module and ScriptModule.



In [37]:
criterion = nn.CrossEntropyLoss()

loss = criterion(logits, labels)

NameError: name 'logits' is not defined

In [40]:

num_samples = X_padded.size(0)
indices = torch.randperm(num_samples)

split = int(0.8 * num_samples)
train_idx, test_idx = indices[:split], indices[split:]

X_train = X_padded[train_idx]
X_test = X_padded[test_idx]

lengths_train = lengths[train_idx]
lengths_test = lengths[test_idx]


In [61]:
Y_enc = Y.apply(lambda x: 1 if str(x).strip().lower() =='positive' else 0)
Y_tensor = torch.tensor(Y_enc.values, dtype=torch.long)

In [62]:
y_train = Y_tensor[train_idx]
y_test = Y_tensor[test_idx]